# Nestlé HR Policy RAG Assistant

Simple retrieval-augmented generation over Nestlé HR policy documents using:
- **PyPDFLoader** — document ingestion
- **ChromaDB + OpenAI Embeddings** — vector storage and semantic search
- **GPT-3.5 Turbo** — answer generation

## Step 1 — Install Dependencies

In [ ]:
# !pip install openai langchain langchain-openai langchain-community langchain-chroma chromadb pypdf python-dotenv tiktoken

## Step 2 — Import Libraries

In [ ]:
import os
import gradio as gr
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found. Please check your .env file.")

print("Environment loaded successfully.")

## Step 3 — Load and Split the HR Policy PDF

In [ ]:
PDF_PATH = "nestle_hr_policy.pdf"

if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(f"'{PDF_PATH}' not found. Place it in the same directory as this notebook.")

loader = PyPDFLoader(PDF_PATH)
raw_pages = loader.load()
print(f"Loaded {len(raw_pages)} pages.")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""]
)
chunks = splitter.split_documents(raw_pages)
print(f"Split into {len(chunks)} chunks.")

In [ ]:
# --- Inspect raw extracted text (run once to verify PDF content) ---
for i, page in enumerate(raw_pages[:5]):
    print(f"\n=== Page {i+1} ===")
    print(page.page_content[:500])
    print("...")

## Step 4 — Create Vector Store

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=OPENAI_API_KEY)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
)
print(f"Created in-memory vector store with {vectorstore._collection.count()} embeddings.")

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 6, "fetch_k": 20}
)

## Step 5 — Build the RAG Chain

In [ ]:
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0, api_key=OPENAI_API_KEY)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a knowledgeable and professional HR assistant for Nestlé. "
     "Use the HR policy excerpts below to answer the employee's question. "
     "Synthesize and summarize the relevant information — do not require a verbatim match. "
     "Be concise, accurate, and empathetic. "
     "Only if the topic is genuinely absent from the excerpts, say: "
     "'I'm sorry, I couldn't find that information in Nestlé's HR policy documents. "
     "Please contact the HR department directly for assistance.'"
     "\n\nContext:\n{context}"),
    ("human", "{input}"),
])

rag_chain = create_retrieval_chain(
    retriever,
    create_stuff_documents_chain(llm, qa_prompt)
)

print("RAG chain ready.")

In [ ]:
# --- Debug: inspect retrieved chunks for a query ---
def debug_context(question: str):
    result = rag_chain.invoke({"input": question})
    print(f"ANSWER: {result['answer']}\n")
    print("=" * 60)
    print("RETRIEVED CHUNKS:")
    for i, doc in enumerate(result["context"], 1):
        page = doc.metadata.get("page", "?")
        print(f"\n[Chunk {i} | page {page + 1}]\n{doc.page_content}")
        print("-" * 40)

debug_context("What is Nestlé's policy on remote work?")

## Step 6 — Gradio Chat Interface

In [ ]:
def answer_query(user_message: str, history: list) -> str:
    result = rag_chain.invoke({"input": user_message})
    answer = result["answer"]
    sources = result.get("context", [])
    if sources:
        pages = sorted({doc.metadata.get("page", 0) + 1 for doc in sources})
        answer += f"\n\n*Source page(s): {', '.join(str(p) for p in pages)}*"
    return answer


demo = gr.ChatInterface(
    fn=answer_query,
    title="Nestlé HR Policy Assistant",
    description="Ask any question about Nestlé's HR policies. Answers are grounded in the official policy documents.",
    examples=[
        "What is the annual leave entitlement for employees?",
        "What is Nestlé's policy on remote work?",
        "How does Nestlé handle workplace harassment complaints?",
        "What are the performance review processes?",
        "What health and wellness benefits does Nestlé offer?",
    ],
)

In [ ]:
demo.launch(theme=gr.themes.Soft(), share=True)